# cv-pilznet – Training auf Google Colab

Dünner Runner-Notebook: holt Repo-Code und Datensatz, ruft dann die eigentliche Logik aus `src/` auf (Datenpipeline, Training, Auswertung liegen als normale Python-Module im Repo, nicht hier im Notebook).

**Vorher sicherstellen:** Laufzeittyp auf GPU stellen (Laufzeit -> Laufzeittyp ändern -> GPU/T4).

In [ ]:
!nvidia-smi

## 1. Repo-Code holen

Das GitLab-Repo ist privat. Entweder ein Personal Access Token in die URL einsetzen (`https://oauth2:<TOKEN>@gitlab.bht-berlin.de/...`), oder alternativ den `src/`-Ordner manuell über das Colab-Dateiupload-Icon hochladen und diese Zelle überspringen.

In [ ]:
GITLAB_TOKEN = ""  # Personal Access Token hier eintragen (read_repository reicht)
REPO_URL = "https://gitlab.bht-berlin.de/s84985/cv-pilznet.git"
clone_url = REPO_URL.replace("https://", f"https://oauth2:{GITLAB_TOKEN}@") if GITLAB_TOKEN else REPO_URL

!git clone {clone_url}
%cd cv-pilznet

## 2. Abhängigkeiten

Torch/torchvision/sklearn/pandas/matplotlib sind in Colab vorinstalliert, nur `albumentations` und die `kaggle`-CLI fehlen.

In [ ]:
!pip install -q albumentations kaggle

## 3. Datensatz laden

`kaggle.json` (API-Token, siehe README.md im Repo für die Einrichtung) hier hochladen, dann Datensatz herunterladen und entpacken.

In [ ]:
from google.colab import files

uploaded = files.upload()  # kaggle.json auswählen
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d zlatan599/mushroom1 -p data/raw --unzip

## 4. Labels aufbereiten (Phase 1)

In [ ]:
!python -m src.build_labels

## 5. Training (Phase 3)

Beide Architekturen für den späteren Vergleich trainieren. Parameter bei Bedarf anpassen (z. B. `--epochs`, `--batch-size`).

In [ ]:
!python -m src.train --arch mobilenet_v2 --epochs 15 --freeze-epochs 3 --batch-size 64

In [ ]:
!python -m src.train --arch resnet50 --epochs 15 --freeze-epochs 3 --batch-size 64

## 6. Ergebnisse herunterladen

Checkpoints (`models/*.pt`) und Trainingshistorien (`results/*_history.json`) lokal herunterladen, um `src.evaluate` (Confusion Matrices) lokal auf der CPU laufen zu lassen — das ist für reine Inferenz schnell genug und braucht keine Colab-GPU-Zeit.

In [ ]:
!zip -r run_output.zip models results
from google.colab import files
files.download("run_output.zip")